In [ ]:
# Cell 1: Imports
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display
import ipywidgets as widgets 

torch.manual_seed(0)

# Cell 2: Define the neural network with Parametric Tanh and Initialization options
class ParametricTanh(nn.Module):
    def __init__(self, initial_alpha=1.0):
        super(ParametricTanh, self).__init__()
        self.alpha = nn.Parameter(torch.tensor(initial_alpha, dtype=torch.float32))

    def forward(self, x):
        return torch.tanh(self.alpha * x)

class FCNN(nn.Module):
    def __init__(self, layers, activation_type='Tanh', initial_alpha=1.0, initialization_method='default'):
        super(FCNN, self).__init__()
        if activation_type == 'ParametricTanh':
            self.activation = ParametricTanh(initial_alpha=initial_alpha)
        elif activation_type == 'Tanh':
            self.activation = nn.Tanh()
        else:
            raise ValueError("Unsupported activation type. Choose 'Tanh' or 'ParametricTanh'.")
            
        self.layers = nn.ModuleList()
        for i in range(len(layers) - 1):
            self.layers.append(nn.Linear(layers[i], layers[i + 1]))
        
        self._initialize_weights(initialization_method)

    def _initialize_weights(self, method):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                if method == 'xavier_uniform':
                    nn.init.xavier_uniform_(m.weight)
                    if m.bias is not None:
                        nn.init.constant_(m.bias, 0)
                elif method == 'xavier_normal':
                    nn.init.xavier_normal_(m.weight)
                    if m.bias is not None:
                        nn.init.constant_(m.bias, 0)
                elif method == 'kaiming_uniform': 
                    nonlinearity = 'tanh' 
                    if isinstance(self.activation, nn.LeakyReLU) or isinstance(self.activation, nn.ReLU):
                        nonlinearity = 'relu'
                    nn.init.kaiming_uniform_(m.weight, mode='fan_in', nonlinearity=nonlinearity)
                    if m.bias is not None:
                        nn.init.constant_(m.bias, 0)
                elif method == 'kaiming_normal':
                    nonlinearity = 'tanh' 
                    if isinstance(self.activation, nn.LeakyReLU) or isinstance(self.activation, nn.ReLU):
                        nonlinearity = 'relu'
                    nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity=nonlinearity)
                    if m.bias is not None:
                        nn.init.constant_(m.bias, 0)
                elif method == 'normal':
                    nn.init.normal_(m.weight, mean=0, std=0.01) 
                    if m.bias is not None:
                        nn.init.constant_(m.bias, 0)
                elif method == 'uniform':
                    nn.init.uniform_(m.weight, a=-0.1, b=0.1) 
                    if m.bias is not None:
                        nn.init.constant_(m.bias, 0)
                elif method == 'default':
                    pass 
                else:
                    raise ValueError(f"Unsupported initialization method: {method}")

    def forward(self, x, t):
        inputs = torch.cat((x, t), dim=1)
        for i in range(len(self.layers) - 1):
            inputs = self.activation(self.layers[i](inputs))
        return self.layers[-1](inputs)

# Cell 3: Define PDE residual
def pde_residual(model, x, t, alpha_pde): 
    x.requires_grad_(True)
    t.requires_grad_(True)
    u = model(x, t)

    u_t = torch.autograd.grad(u, t, grad_outputs=torch.ones_like(u, device=x.device), create_graph=True)[0]
    u_x = torch.autograd.grad(u, x, grad_outputs=torch.ones_like(u, device=x.device), create_graph=True)[0]
    u_xx = torch.autograd.grad(u_x, x, grad_outputs=torch.ones_like(u, device=x.device), create_graph=True)[0]

    return u_t - alpha_pde * u_xx

# Cell 4: Training setup
alpha_pde = 0.01 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Global variables to be controlled by "widgets"
current_activation_type = 'Tanh'
current_initial_alpha_param_tanh = 1.0 
current_initialization_method = 'default' 
current_learning_rate = 1e-3
current_epochs = 5000
current_hidden_layers_size = 64
current_num_hidden_layers = 3 
current_seed = 0 
current_optimizer_type = 'Adam' # Default optimizer

def initialize_and_train_model(
    activation_type='Tanh', 
    initial_alpha_param_tanh=1.0, 
    initialization_method='default', 
    learning_rate=1e-3, 
    epochs=5000, 
    hidden_layers_size=64,
    num_hidden_layers=3, 
    seed=0,
    optimizer_type='Adam' 
):
    global model, optimizer, x_f, t_f, x_i, t_i, u_i, x_b0, x_b1, t_b, u_b0, u_b1
    
    # Use the output widget to display print statements
    with output_area: 
        output_area.clear_output() # Clear previous output
        print(f"\n--- Starting training with parameters ---")
        print(f"Activation: {activation_type}")
        if activation_type == 'ParametricTanh':
            print(f"Parametric Tanh Initial Alpha: {initial_alpha_param_tanh}")
        print(f"Initialization Method: {initialization_method}")
        print(f"Optimizer: {optimizer_type}") 
        print(f"Learning Rate: {learning_rate}")
        print(f"Epochs: {epochs}")
        print(f"Hidden Layers Size: {hidden_layers_size}")
        print(f"Number of Hidden Layers: {num_hidden_layers}")
        print(f"Seed: {seed}")
        print(f"----------------------------------------")

        torch.manual_seed(seed) 

        # Dynamically create the layers list
        model_layers = [2] + [hidden_layers_size] * num_hidden_layers + [1]
        model = FCNN(model_layers, 
                     activation_type=activation_type, 
                     initial_alpha=initial_alpha_param_tanh,
                     initialization_method=initialization_method).to(device)

        # Collocation points
        N_f = 10000
        x_f = torch.rand((N_f, 1), device=device)
        t_f = torch.rand((N_f, 1), device=device)

        # Initial condition
        N_i = 100
        x_i = torch.linspace(0, 1, N_i).view(-1, 1).to(device)
        t_i = torch.zeros_like(x_i).to(device)
        u_i = torch.sin(np.pi * x_i).to(device)

        # Boundary conditions
        N_b = 100
        t_b = torch.linspace(0, 1, N_b).view(-1, 1).to(device)
        x_b0 = torch.zeros_like(t_b).to(device)
        x_b1 = torch.ones_like(t_b).to(device)
        u_b0 = torch.zeros_like(t_b).to(device)
        u_b1 = torch.zeros_like(t_b).to(device)

        # Optimizer selection
        if optimizer_type == 'Adam':
            optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
        elif optimizer_type == 'SGD':
            optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, momentum=0.9) 
        elif optimizer_type == 'AdamW':
            optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
        else:
            raise ValueError(f"Unsupported optimizer type: {optimizer_type}")


        # List to store loss values for plotting
        loss_history = []

        # Cell 5: Training loop
        for epoch in range(epochs):
            optimizer.zero_grad()
            
            # Loss: PDE residual
            f_res = pde_residual(model, x_f, t_f, alpha_pde)
            loss_f = torch.mean(f_res**2)

            # Loss: initial condition
            u_pred_i = model(x_i, t_i)
            loss_i = torch.mean((u_pred_i - u_i)**2)

            # Loss: boundary condition
            u_pred_b0 = model(x_b0, t_b)
            u_pred_b1 = model(x_b1, t_b)
            loss_b = torch.mean((u_pred_b0 - u_b0)**2) + torch.mean((u_pred_b1 - u_b1)**2)

            loss = loss_f + loss_i + loss_b
            loss.backward()
            optimizer.step()

            loss_history.append(loss.item()) # Store the loss

            if epoch % 500 == 0 or epoch == epochs -1: 
                print(f"Epoch {epoch}, Loss: {loss.item():.5e}")
        
        print("Training complete.")

        # Plotting the MSE Loss vs. Epochs
        fig_loss, ax_loss = plt.subplots(figsize=(8, 5))
        ax_loss.plot(range(epochs), loss_history, label='Total MSE Loss')
        ax_loss.set_xlabel('Epoch')
        ax_loss.set_ylabel('Loss (MSE)')
        ax_loss.set_title('Training Loss vs. Epochs')
        ax_loss.set_yscale('log') # Log scale is often useful for loss curves
        ax_loss.legend()
        ax_loss.grid(True, which="both", ls="--", c='0.7')
        display(fig_loss) # Display the plot in the output area
        plt.close(fig_loss) # Close the figure to prevent duplicate display

        # After training, run comparison and animation
        run_comparison_and_animation()

# Cell 7: Compare with analytical solution and Animate the solution (moved into a function)
def run_comparison_and_animation():
    with output_area: # Ensure animation output is also captured
        print("\n--- Generating results and animation ---")
        x_test = torch.linspace(0, 1, 100).view(-1, 1).to(device)
        t_test = torch.linspace(0, 1, 100).view(-1, 1).to(device)

        X, T = torch.meshgrid(x_test.squeeze(), t_test.squeeze(), indexing='ij')
        x_flat = X.reshape(-1, 1).to(device)
        t_flat = T.reshape(-1, 1).to(device)

        with torch.no_grad():
            u_pred = model(x_flat, t_flat).cpu().numpy().reshape(100, 100)
            u_exact = np.exp(-np.pi**2 * alpha_pde * T.cpu().numpy()) * np.sin(np.pi * X.cpu().numpy())
            error = np.linalg.norm(u_pred - u_exact) / np.linalg.norm(u_exact)
            print(f"Relative L2 error: {error:.2e}")

        fig, ax = plt.subplots()
        line1, = ax.plot(x_test.cpu(), u_pred[:, 0], label='PINN')
        line2, = ax.plot(x_test.cpu(), u_exact[:, 0], '--', label='Exact')
        ax.set_ylim(-0.1, 1.1)  
        ax.legend()
        ax.set_xlabel('x')
        ax.set_ylabel('u(x,t)')
        plt.close(fig) 

        def animate(i):
            line1.set_ydata(u_pred[:, i])
            line2.set_ydata(u_exact[:, i])
            ax.set_title(f't = {t_test[i].item():.2f}')
            return line1, line2

        ani = animation.FuncAnimation(fig, animate, frames=100, interval=100)
        display(HTML(ani.to_jshtml()))

# --- Widget-like Interaction (Manual Simulation or ipywidgets) ---
if 'widgets' in globals(): 
    activation_dropdown = widgets.Dropdown(
        options=['Tanh', 'ParametricTanh'],
        value=current_activation_type,
        description='Activation:',
        disabled=False,
    )
    initial_alpha_slider = widgets.FloatSlider(
        min=0.1, max=5.0, step=0.1, value=current_initial_alpha_param_tanh,
        description='Parametric Tanh Alpha:',
        continuous_update=False
    )
    initialization_dropdown = widgets.Dropdown(
        options=['default', 'xavier_uniform', 'xavier_normal', 'kaiming_uniform', 'kaiming_normal', 'normal', 'uniform'],
        value=current_initialization_method,
        description='Init Method:',
        disabled=False,
    )
    optimizer_dropdown = widgets.Dropdown(
        options=['Adam', 'SGD', 'AdamW'],
        value=current_optimizer_type,
        description='Optimizer:',
        disabled=False,
    )
    
    # Text box for learning rate
    lr_text = widgets.FloatText(
        value=current_learning_rate,
        description='Learning Rate:',
        disabled=False
    )
    
    epochs_text = widgets.IntText( 
        value=current_epochs,
        description='Epochs:',
        disabled=False
    )
    hidden_size_text = widgets.IntText( 
        value=current_hidden_layers_size,
        description='Hidden Size:',
        disabled=False
    )
    num_hidden_layers_text = widgets.IntText( 
        value=current_num_hidden_layers,
        description='Num Hidden Layers:',
        disabled=False
    )

    seed_int_text = widgets.IntText(
        value=current_seed,
        description='Seed:',
        disabled=False
    )
    train_button = widgets.Button(description="Train Model")
    
    output_area = widgets.Output()

    def on_train_button_clicked(b):
        initialize_and_train_model(
            activation_type=activation_dropdown.value,
            initial_alpha_param_tanh=initial_alpha_slider.value,
            initialization_method=initialization_dropdown.value,
            optimizer_type=optimizer_dropdown.value, 
            learning_rate=lr_text.value, # Use value from text box
            epochs=epochs_text.value, 
            hidden_layers_size=hidden_size_text.value, 
            num_hidden_layers=num_hidden_layers_text.value, 
            seed=seed_int_text.value
        )

    train_button.on_click(on_train_button_clicked)

    # Display widgets, including the output_area
    display(widgets.VBox([
        activation_dropdown, initial_alpha_slider, initialization_dropdown,
        optimizer_dropdown, 
        lr_text, # Display the learning rate text box
        epochs_text, 
        hidden_size_text, 
        num_hidden_layers_text, 
        seed_int_text, train_button,
        output_area 
    ]))
else:
    print("ipywidgets not imported. To use interactive widgets, run this in a Jupyter environment and uncomment 'import ipywidgets as widgets'.")
    print("\nRunning a single training example now (adjust parameters in the script to try different settings):")
    
    class DummyOutput:
        def __enter__(self):
            pass
        def __exit__(self, exc_type, exc_val, exc_tb):
            pass
        def clear_output(self):
            pass
    
    output_area = DummyOutput() 

    initialize_and_train_model(epochs=3000, learning_rate=2e-3, 
                               activation_type='ParametricTanh', initial_alpha_param_tanh=1.5, 
                               initialization_method='xavier_normal', 
                               num_hidden_layers=2, 
                               optimizer_type='Adam', 
                               seed=42)